# 集群上svaba使用流程

## 一、数据来源

### 197个样本路径：/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/PDAC_WGS/wgs_197_bams/

### 80组配对样本名称列表：/mnt/home/ygjx/chenkejin/delly/delly-main/task_list.txt

## 二、配置环境及编译

### 1、创建环境

In [ ]:
conda create -n svaba_env -y -c conda-forge -c bioconda \
    cmake>=3.14 \
    gcc cxx-compiler \
    htslib zlib bzip2 xz \
    jemalloc sqlite

conda activate svaba_env

conda install samtools -c bioconda -y

### 2、下载svaba源码并编译 

In [ ]:
cd /mnt/home/ygjx/chenkejin/manta/

git clone --recursive https://github.com/walaj/svaba
cd svaba

sed -i '/#include <memory>/a #include <cstdint>' /mnt/home/ygjx/chenkejin/svaba/src/svaba/LearnBamParams.h

mkdir -p build && cd build
cmake .. -DUSE_JEMALLOC=ON

make -j 8

### 3、svaba要求目录下有一整套专属的 BWA 索引文件（通常包含 .bwt, .pac, .ann, .amb, .sa 这 5 个隐藏文件），可用下面的脚本构建

In [ ]:
#!/bin/bash
#SBATCH --job-name=bwa_index
#SBATCH --nodes=1
#SBATCH --cpus-per-task=4
#SBATCH --mem=16G
#SBATCH --output=bwa_index.log

source /mnt/home/ygjx/chenkejin/anaconda3/etc/profile.d/conda.sh
conda activate svaba_env

echo "开始构建 BWA 索引 (预计需要 1-2 小时)..."
bwa index /mnt/home/ygjx/chenkejin/delly/delly-main/Homo_sapiens_assembly38.fasta
echo "构建完成！"

## 三、运行

In [ ]:
mkdir -p /mnt/home/ygjx/chenkejin/SvABA
cd /mnt/home/ygjx/chenkejin/SvABA

### 1、单样本运行

### 脚本路径：/mnt/home/ygjx/chenkejin/SvABA/test_single_svaba.sh

### bash test_single_svaba.sh --normal 1866277N --tumor 1866277T --prefix 1866277

In [ ]:
#!/bin/bash
#SBATCH --job-name=svaba_single_pdac
#SBATCH --nodes=1
#SBATCH --cpus-per-task=16
#SBATCH --mem=48G
#SBATCH --output=/mnt/home/ygjx/chenkejin/SvABA/logs/test_single_svaba_%j.out
#SBATCH --error=/mnt/home/ygjx/chenkejin/SvABA/logs/test_single_svaba_%j.err

set -euo pipefail

source /mnt/home/ygjx/chenkejin/anaconda3/etc/profile.d/conda.sh
conda activate svaba

REF_FA="${REF_FA:-/mnt/home/ygjx/chenkejin/delly/delly-main/Homo_sapiens_assembly38.fasta}"
MANIFEST="${MANIFEST:-/mnt/home/ygjx/chenkejin/bam_qc/PDAC_WGS_full_BAM_QC/pdac_197_bam_manifest.tsv}"
WORK_DIR="${WORK_DIR:-/mnt/home/ygjx/chenkejin/SvABA}"
THREADS="${THREADS:-${SLURM_CPUS_PER_TASK:-16}}"

DBSNP_VCF="${DBSNP_VCF:-}"
REGION_BED="${REGION_BED:-}"

NORMAL_ID="1866277N"
TUMOR_ID="1866277T"
PREFIX=""

usage() {
  cat <<'EOF'
Usage:
  bash test_single_svaba.sh [--normal NORMAL_ID] [--tumor TUMOR_ID] [--prefix PREFIX]

Defaults:
  --normal 1866277N
  --tumor  1866277T

Optional environment variables:
  REF_FA      Default: /mnt/home/ygjx/chenkejin/delly/delly-main/Homo_sapiens_assembly38.fasta
  MANIFEST    Default: /mnt/home/ygjx/chenkejin/bam_qc/PDAC_WGS_full_BAM_QC/pdac_197_bam_manifest.tsv
  WORK_DIR    Default: /mnt/home/ygjx/chenkejin/SvABA
  THREADS     Default: SLURM_CPUS_PER_TASK or 16
  DBSNP_VCF   Optional dbSNP VCF, passed to svaba run with -D
  REGION_BED  Optional region BED, passed to svaba run with -k
EOF
}

while [ "$#" -gt 0 ]; do
  case "$1" in
    --normal)
      NORMAL_ID="${2:?ERROR: --normal needs a sample id}"
      shift 2
      ;;
    --tumor)
      TUMOR_ID="${2:?ERROR: --tumor needs a sample id}"
      shift 2
      ;;
    --prefix)
      PREFIX="${2:?ERROR: --prefix needs a value}"
      shift 2
      ;;
    --manifest)
      MANIFEST="${2:?ERROR: --manifest needs a path}"
      shift 2
      ;;
    --ref)
      REF_FA="${2:?ERROR: --ref needs a reference fasta}"
      shift 2
      ;;
    --work-dir)
      WORK_DIR="${2:?ERROR: --work-dir needs a path}"
      shift 2
      ;;
    -h|--help)
      usage
      exit 0
      ;;
    *)
      echo "ERROR: unknown argument: $1" >&2
      usage >&2
      exit 1
      ;;
  esac
done

if [ -z "${PREFIX}" ]; then
  PREFIX="${NORMAL_ID%N}"
fi

MASTER_LOG="${WORK_DIR}/master_progress_svaba_pdac197.log"
mkdir -p "${WORK_DIR}/logs"
SAMPLE_LOG="${WORK_DIR}/logs/${PREFIX}.test_single_svaba.log"
exec > >(tee -i "${SAMPLE_LOG}") 2>&1

echo "=========================================================="
echo "[$(date '+%Y-%m-%d %H:%M:%S')] [START] SvABA single test: ${PREFIX}"
echo "Normal: ${NORMAL_ID}"
echo "Tumor : ${TUMOR_ID}"
echo "Manifest: ${MANIFEST}"
echo "Reference: ${REF_FA}"
echo "Threads: ${THREADS}"
echo "Expected outputs: *.svaba.somatic.sv.vcf and *.svaba.somatic.indel.vcf"
echo "=========================================================="
echo "[$(date '+%Y-%m-%d %H:%M:%S')] [START] Single test ${PREFIX}" >> "${MASTER_LOG}"

if [ ! -f "${MANIFEST}" ]; then
  echo "ERROR: manifest not found: ${MANIFEST}" >&2
  exit 1
fi

if [ ! -f "${REF_FA}" ]; then
  echo "ERROR: reference fasta not found: ${REF_FA}" >&2
  exit 1
fi

if [ ! -f "${REF_FA}.fai" ]; then
  echo "ERROR: reference fasta index not found: ${REF_FA}.fai" >&2
  exit 1
fi

get_manifest_field() {
  local sample="$1"
  local field="$2"
  awk -F'\t' -v sample="${sample}" -v field="${field}" '
    NR==1 {
      for (i=1; i<=NF; i++) col[$i]=i
      if (!("sample" in col) || !(field in col)) exit 2
      next
    }
    $(col["sample"]) == sample {
      print $(col[field])
      found=1
      exit
    }
    END {
      if (!found) exit 1
    }
  ' "${MANIFEST}"
}

NORMAL_BAM="$(get_manifest_field "${NORMAL_ID}" "bam")"
TUMOR_BAM="$(get_manifest_field "${TUMOR_ID}" "bam")"

if [ ! -f "${NORMAL_BAM}" ] || [ ! -f "${TUMOR_BAM}" ]; then
  echo "ERROR: BAM missing after manifest lookup." >&2
  echo "Normal BAM: ${NORMAL_BAM}" >&2
  echo "Tumor BAM : ${TUMOR_BAM}" >&2
  exit 1
fi

OPTIONAL_ARGS=()
if [ -n "${DBSNP_VCF}" ]; then
  if [ ! -f "${DBSNP_VCF}" ]; then
    echo "ERROR: DBSNP_VCF was set but file not found: ${DBSNP_VCF}" >&2
    exit 1
  fi
  OPTIONAL_ARGS+=("-D" "${DBSNP_VCF}")
fi

if [ -n "${REGION_BED}" ]; then
  if [ ! -f "${REGION_BED}" ]; then
    echo "ERROR: REGION_BED was set but file not found: ${REGION_BED}" >&2
    exit 1
  fi
  OPTIONAL_ARGS+=("-k" "${REGION_BED}")
fi

FINAL_VCF_DIR="${WORK_DIR}/PDAC_197_single_test_result"
mkdir -p "${FINAL_VCF_DIR}"

final_sv_vcf="${FINAL_VCF_DIR}/${PREFIX}.svaba.somatic.sv.vcf"
final_indel_vcf="${FINAL_VCF_DIR}/${PREFIX}.svaba.somatic.indel.vcf"

RUN_DIR="${WORK_DIR}/sandbox/${PREFIX}_svaba_run"
rm -rf "${RUN_DIR}"
mkdir -p "${RUN_DIR}"
cd "${RUN_DIR}"

echo "Normal BAM: ${NORMAL_BAM}"
echo "Tumor BAM : ${TUMOR_BAM}"

{
  echo "[$(date '+%Y-%m-%d %H:%M:%S')] Run svaba run, HCC1395-validated mode..."
  svaba run \
    -t "${TUMOR_BAM}" \
    -n "${NORMAL_BAM}" \
    -p "${THREADS}" \
    -a "${PREFIX}" \
    -G "${REF_FA}" \
    "${OPTIONAL_ARGS[@]}"

  if [ -f "${PREFIX}.svaba.somatic.sv.vcf" ] && [ -f "${PREFIX}.svaba.somatic.indel.vcf" ]; then
    cp "${PREFIX}.svaba.somatic.sv.vcf" "${final_sv_vcf}"
    cp "${PREFIX}.svaba.somatic.indel.vcf" "${final_indel_vcf}"

    cd "${WORK_DIR}"
    rm -rf "${RUN_DIR}"

    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SUCCESS] ${PREFIX}"
    echo "Final SV VCF: ${final_sv_vcf}"
    echo "Final indel VCF: ${final_indel_vcf}"
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SUCCESS] Single test ${PREFIX}" >> "${MASTER_LOG}"
  else
    echo "ERROR: SvABA finished but expected somatic VCF files were not found." >&2
    echo "Expected: ${PREFIX}.svaba.somatic.sv.vcf" >&2
    echo "Expected: ${PREFIX}.svaba.somatic.indel.vcf" >&2
    false
  fi
} || {
  echo "[$(date '+%Y-%m-%d %H:%M:%S')] [ERROR] ${PREFIX} failed" >> "${MASTER_LOG}"
  exit 1
}


### 2、批量处理

### 脚本路径：/mnt/home/ygjx/chenkejin/SvABA/batch_svaba.sh

In [ ]:
#!/bin/bash
#SBATCH --job-name=SvABA_PDAC197
#SBATCH --nodes=1
#SBATCH --cpus-per-task=16
#SBATCH --mem=48G
#SBATCH --output=/mnt/home/ygjx/chenkejin/SvABA/logs/slurm_array_%A_%a.out
#SBATCH --error=/mnt/home/ygjx/chenkejin/SvABA/logs/slurm_array_%A_%a.err

set -euo pipefail

source /mnt/home/ygjx/chenkejin/anaconda3/etc/profile.d/conda.sh
conda activate svaba

REF_FA="${REF_FA:-/mnt/home/ygjx/chenkejin/delly/delly-main/Homo_sapiens_assembly38.fasta}"
MANIFEST="${MANIFEST:-/mnt/home/ygjx/chenkejin/bam_qc/PDAC_WGS_full_BAM_QC/pdac_197_bam_manifest.tsv}"
TASK_LIST="${TASK_LIST:-/mnt/home/ygjx/chenkejin/delly/delly-main/task_list.txt}"
WORK_DIR="${WORK_DIR:-/mnt/home/ygjx/chenkejin/SvABA}"
THREADS="${THREADS:-${SLURM_CPUS_PER_TASK:-16}}"

DBSNP_VCF="${DBSNP_VCF:-}"
REGION_BED="${REGION_BED:-}"

MASTER_LOG="${WORK_DIR}/master_progress_svaba_pdac197.log"

if [ -z "${SLURM_ARRAY_TASK_ID:-}" ]; then
  echo "ERROR: this script must be submitted as a Slurm array job." >&2
  echo "Example:" >&2
  echo "  N=\\$(awk 'NF>=2 && \\$1 !~ /^#/{n++} END{print n+0}' ${TASK_LIST})" >&2
  echo "  sbatch --array=1-\\${N}%10 batch_svaba.sh" >&2
  exit 1
fi

if [ ! -f "${MANIFEST}" ]; then
  echo "ERROR: manifest not found: ${MANIFEST}" >&2
  exit 1
fi

if [ ! -f "${TASK_LIST}" ]; then
  echo "ERROR: task list not found: ${TASK_LIST}" >&2
  echo "Expected format per non-comment line: NORMAL_ID<TAB_or_SPACE>TUMOR_ID" >&2
  exit 1
fi

if [ ! -f "${REF_FA}" ]; then
  echo "ERROR: reference fasta not found: ${REF_FA}" >&2
  exit 1
fi

if [ ! -f "${REF_FA}.fai" ]; then
  echo "ERROR: reference fasta index not found: ${REF_FA}.fai" >&2
  exit 1
fi

TOTAL_TASKS="$(awk 'NF>=2 && $1 !~ /^#/{n++} END{print n+0}' "${TASK_LIST}")"
LINE="$(awk -v idx="${SLURM_ARRAY_TASK_ID}" '
  NF>=2 && $1 !~ /^#/ {
    n++
    if (n==idx) {
      print $1 "\t" $2
      exit
    }
  }
' "${TASK_LIST}" | tr -d '\r')"

if [ -z "${LINE}" ]; then
  echo "ERROR: empty task line for SLURM_ARRAY_TASK_ID=${SLURM_ARRAY_TASK_ID}" >&2
  echo "Total usable tasks in ${TASK_LIST}: ${TOTAL_TASKS}" >&2
  exit 1
fi

IFS=$'\t' read -r NORMAL_ID TUMOR_ID <<< "${LINE}"
PREFIX="${NORMAL_ID%N}"

mkdir -p "${WORK_DIR}/logs"
SAMPLE_LOG="${WORK_DIR}/logs/${PREFIX}.svaba.log"
exec > >(tee -i "${SAMPLE_LOG}") 2>&1

echo "=========================================================="
echo "[$(date '+%Y-%m-%d %H:%M:%S')] [START] SvABA PDAC sample pair: ${PREFIX}"
echo "Task: ${SLURM_ARRAY_TASK_ID}/${TOTAL_TASKS}"
echo "Node: $(hostname)"
echo "Normal: ${NORMAL_ID}"
echo "Tumor : ${TUMOR_ID}"
echo "Manifest: ${MANIFEST}"
echo "Task list: ${TASK_LIST}"
echo "Reference: ${REF_FA}"
echo "Threads: ${THREADS}"
echo "Expected outputs: *.svaba.somatic.sv.vcf and *.svaba.somatic.indel.vcf"
echo "=========================================================="
echo "[$(date '+%Y-%m-%d %H:%M:%S')] [START] Task ${SLURM_ARRAY_TASK_ID}/${TOTAL_TASKS}: ${PREFIX}" >> "${MASTER_LOG}"

get_manifest_field() {
  local sample="$1"
  local field="$2"
  awk -F'\t' -v sample="${sample}" -v field="${field}" '
    NR==1 {
      for (i=1; i<=NF; i++) col[$i]=i
      if (!("sample" in col) || !(field in col)) exit 2
      next
    }
    $(col["sample"]) == sample {
      print $(col[field])
      found=1
      exit
    }
    END {
      if (!found) exit 1
    }
  ' "${MANIFEST}"
}

NORMAL_BAM="$(get_manifest_field "${NORMAL_ID}" "bam")"
TUMOR_BAM="$(get_manifest_field "${TUMOR_ID}" "bam")"

if [ ! -f "${NORMAL_BAM}" ] || [ ! -f "${TUMOR_BAM}" ]; then
  echo "[$(date '+%Y-%m-%d %H:%M:%S')] [ERROR] BAM missing after manifest lookup."
  echo "Normal BAM: ${NORMAL_BAM}"
  echo "Tumor BAM : ${TUMOR_BAM}"
  echo "[$(date '+%Y-%m-%d %H:%M:%S')] [ERROR] Task ${SLURM_ARRAY_TASK_ID}/${TOTAL_TASKS}: ${PREFIX} missing BAM" >> "${MASTER_LOG}"
  exit 1
fi

OPTIONAL_ARGS=()
if [ -n "${DBSNP_VCF}" ]; then
  if [ ! -f "${DBSNP_VCF}" ]; then
    echo "ERROR: DBSNP_VCF was set but file not found: ${DBSNP_VCF}" >&2
    exit 1
  fi
  OPTIONAL_ARGS+=("-D" "${DBSNP_VCF}")
fi

if [ -n "${REGION_BED}" ]; then
  if [ ! -f "${REGION_BED}" ]; then
    echo "ERROR: REGION_BED was set but file not found: ${REGION_BED}" >&2
    exit 1
  fi
  OPTIONAL_ARGS+=("-k" "${REGION_BED}")
fi

FINAL_VCF_DIR="${WORK_DIR}/PDAC_197_result"
mkdir -p "${FINAL_VCF_DIR}"

final_sv_vcf="${FINAL_VCF_DIR}/${PREFIX}.svaba.somatic.sv.vcf"
final_indel_vcf="${FINAL_VCF_DIR}/${PREFIX}.svaba.somatic.indel.vcf"

if [ -f "${final_sv_vcf}" ] && [ -f "${final_indel_vcf}" ]; then
  echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SKIP] ${PREFIX} already has final SvABA outputs."
  echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SKIP] Task ${SLURM_ARRAY_TASK_ID}/${TOTAL_TASKS}: ${PREFIX}" >> "${MASTER_LOG}"
  exit 0
fi

RUN_DIR="${WORK_DIR}/sandbox/${PREFIX}_svaba_run"
rm -rf "${RUN_DIR}"
mkdir -p "${RUN_DIR}"
cd "${RUN_DIR}"

echo "Normal BAM: ${NORMAL_BAM}"
echo "Tumor BAM : ${TUMOR_BAM}"

{
  echo "[$(date '+%Y-%m-%d %H:%M:%S')] Run svaba run, HCC1395-validated mode..."
  svaba run \
    -t "${TUMOR_BAM}" \
    -n "${NORMAL_BAM}" \
    -p "${THREADS}" \
    -a "${PREFIX}" \
    -G "${REF_FA}" \
    "${OPTIONAL_ARGS[@]}"

  if [ -f "${PREFIX}.svaba.somatic.sv.vcf" ] && [ -f "${PREFIX}.svaba.somatic.indel.vcf" ]; then
    cp "${PREFIX}.svaba.somatic.sv.vcf" "${final_sv_vcf}"
    cp "${PREFIX}.svaba.somatic.indel.vcf" "${final_indel_vcf}"

    cd "${WORK_DIR}"
    rm -rf "${RUN_DIR}"

    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SUCCESS] ${PREFIX}"
    echo "Final SV VCF: ${final_sv_vcf}"
    echo "Final indel VCF: ${final_indel_vcf}"
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] [SUCCESS] Task ${SLURM_ARRAY_TASK_ID}/${TOTAL_TASKS}: ${PREFIX}" >> "${MASTER_LOG}"
  else
    echo "ERROR: SvABA finished but expected somatic VCF files were not found." >&2
    echo "Expected: ${PREFIX}.svaba.somatic.sv.vcf" >&2
    echo "Expected: ${PREFIX}.svaba.somatic.indel.vcf" >&2
    false
  fi
} || {
  echo "[$(date '+%Y-%m-%d %H:%M:%S')] [ERROR] ${PREFIX} failed"
  echo "[$(date '+%Y-%m-%d %H:%M:%S')] [ERROR] Task ${SLURM_ARRAY_TASK_ID}/${TOTAL_TASKS}: ${PREFIX} failed" >> "${MASTER_LOG}"
  exit 1
}


### 批量运行

In [ ]:
TASK_LIST="/mnt/home/ygjx/chenkejin/delly/delly-main/task_list.txt"
N=$(awk 'NF>=2 && $1 !~ /^#/{n++} END{print n+0}' "${TASK_LIST}")

sbatch --array=1-${N}%20 batch_svaba.sh

### 结果路径：/mnt/home/ygjx/chenkejin/SvABA/PDAC_197_result